In [0]:
from pyspark.sql import functions as F

# ---- Config ----
catalog     = "firstdeproject"
schema      = "bronze"
volume_root = f"/Volumes/{catalog}/{schema}/raw_sources"
source_folders = ["source_crm", "source_erp"]

# ---- Discover and load all CSVs ----
for folder in source_folders:
    folder_path = f"{volume_root}/{folder}"

    for f in dbutils.fs.ls(folder_path):
        if not f.name.lower().endswith(".csv"):
            continue

        table_name = f.name[:-4].lower().replace(" ", "_")   # "orders.csv" -> "orders"
        full_table = f"{catalog}.{schema}.{table_name}"

        df = (
            spark.read
            .option("header", True)
            .option("multiLine", True)
            .option("escape", '"')
            .csv(f.path)
            .withColumn("_source_folder", F.lit(folder))
            .withColumn("_source_file", F.lit(f.name))
            .withColumn("_ingested_at", F.current_timestamp())
        )

        (df.write
           .mode("overwrite")                 # use "append" for incremental loads
           .option("overwriteSchema", "true")
           .saveAsTable(full_table))

        print(f"Loaded {df.count()} rows from {folder}/{f.name} -> {full_table}")

In [0]:
%sql
SELECT * FROM firstdeproject.bronze.cust_az12
LIMIT 10;